# PhishGuard AI — End-to-End ML Notebook

This notebook builds the educational phishing detection system used by the Flask website. It uses self-contained synthetic URL and email/message datasets so the project can run without downloading external datasets.

## 1. Problem Statement
Detect phishing risk from either a website URL or suspicious email/message text. The project compares multiple classification models, selects the strongest model using F1-score, and exposes the saved models through a Flask UI.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, ConfusionMatrixDisplay


In [ ]:
url_df = pd.read_csv('../dataset/phishing_urls.csv')
email_df = pd.read_csv('../dataset/phishing_emails.csv')
print(url_df.shape, email_df.shape)
url_df.head()

## 2. Data Cleaning and Inspection

In [ ]:
print(url_df.isnull().sum())
print(email_df.isnull().sum())
print(url_df['phishing'].value_counts())
print(email_df['phishing'].value_counts())

In [ ]:
url_df = url_df.drop_duplicates().dropna()
email_df = email_df.drop_duplicates().dropna()
print('Clean URL rows:', len(url_df))
print('Clean message rows:', len(email_df))

## 3. URL Model Comparison
Models: Logistic Regression, Decision Tree, Random Forest and Gradient Boosting. Metrics: Accuracy, Precision, Recall, F1 and ROC-AUC.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
features=[c for c in url_df.columns if c not in ['url','phishing']]
X=url_df[features]; y=url_df['phishing']
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.2,random_state=42,stratify=y)
models={
'Logistic Regression':Pipeline([('scaler',StandardScaler()),('model',LogisticRegression(max_iter=1000))]),
'Decision Tree':DecisionTreeClassifier(max_depth=10,random_state=42),
'Random Forest':RandomForestClassifier(n_estimators=180,max_depth=14,random_state=42),
'Gradient Boosting':GradientBoostingClassifier(random_state=42)}
results=[]
for name,m in models.items():
    m.fit(Xtr,ytr); p=m.predict(Xte); pr=m.predict_proba(Xte)[:,1]
    results.append([name,accuracy_score(yte,p),precision_score(yte,p),recall_score(yte,p),f1_score(yte,p),roc_auc_score(yte,pr)])
url_results=pd.DataFrame(results,columns=['Model','Accuracy','Precision','Recall','F1','ROC_AUC'])
url_results

In [ ]:
url_results.set_index('Model')[['Accuracy','Precision','Recall','F1']].plot(kind='bar',figsize=(11,5),ylim=(0,1),title='URL Model Comparison'); plt.xticks(rotation=20); plt.tight_layout(); plt.show()

## 4. Email / Message NLP Comparison
TF-IDF converts text into numeric features. We compare Logistic Regression, Linear SVM, Naive Bayes and Random Forest.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
Xe=email_df.message; ye=email_df.phishing
Xtr,Xte,ytr,yte=train_test_split(Xe,ye,test_size=.2,random_state=42,stratify=ye)
text_models={
'Logistic Regression':Pipeline([('tfidf',TfidfVectorizer(ngram_range=(1,2),max_features=6000,stop_words='english')),('model',LogisticRegression(max_iter=1000))]),
'Linear SVM':Pipeline([('tfidf',TfidfVectorizer(ngram_range=(1,2),max_features=6000,stop_words='english')),('model',LinearSVC())]),
'Naive Bayes':Pipeline([('tfidf',TfidfVectorizer(ngram_range=(1,2),max_features=6000,stop_words='english')),('model',MultinomialNB())]),
'Random Forest':Pipeline([('tfidf',TfidfVectorizer(ngram_range=(1,2),max_features=6000,stop_words='english')),('model',RandomForestClassifier(n_estimators=150,random_state=42))])}
results=[]
for name,m in text_models.items():
    m.fit(Xtr,ytr); p=m.predict(Xte); pr=m.predict_proba(Xte)[:,1] if hasattr(m,'predict_proba') else np.where(p==1,.9,.1)
    results.append([name,accuracy_score(yte,p),precision_score(yte,p),recall_score(yte,p),f1_score(yte,p),roc_auc_score(yte,pr)])
text_results=pd.DataFrame(results,columns=['Model','Accuracy','Precision','Recall','F1','ROC_AUC'])
text_results

In [ ]:
text_results.set_index('Model')[['Accuracy','Precision','Recall','F1']].plot(kind='bar',figsize=(11,5),ylim=(0,1),title='Text Model Comparison'); plt.xticks(rotation=20); plt.tight_layout(); plt.show()

## 5. Final Model Selection
The final URL and text models are selected by highest F1-score. The saved Flask application uses these trained pipelines directly; it does not retrain on every request.

In [ ]:
best_url = url_results.sort_values('F1',ascending=False).iloc[0]
best_text = text_results.sort_values('F1',ascending=False).iloc[0]
print('Best URL model:', best_url['Model'])
print('Best Text model:', best_text['Model'])

## 6. Demo Prediction
Paste a URL or message and call the same feature extraction/model logic used by the Flask backend.

In [ ]:
demo_url='https://secure-login.example.com/account/verify'
demo_message='Urgent! Your account is suspended. Verify your password immediately.'
print('Demo URL:', demo_url)
print('Demo message:', demo_message)
print('The Flask app combines the URL risk and message risk into the final risk score.')